
# V2_4 - Statistical figures and tables for the manuscript
Runs **after** `V2_3_Compare_and_Report` (needs only the CSV files in `revised_outputs_v2_REPORT/`; no GPU, about 1 minute).
It rebuilds every generated figure (Figs 5-9, 11) and table (Tables 3-7, S1) cited in the revised manuscript, straight from the report CSVs, so every number in the paper is traceable.
Section 8 is optional: it needs the rasters and recomputes the Fallow->Forest base rate exactly, with a block-bootstrap interval.

Outputs go to `revised_outputs_v2_REPORT/manuscript_figs_tables/` (figures as PNG 300 dpi, vector PDF and 600 dpi TIFF; tables as CSV; one Excel workbook with all tables).

**Print-size design:** every figure is drawn at its final width of 6.3 in (16 cm, the manuscript text width) with 8.5-9.5 pt Times-compatible text, so insert it in Word at 100 % (do not stretch). If Times New Roman is missing in Colab, the cell below installs the metric-identical Liberation Serif.


In [ ]:

# Fonts (Colab): Liberation Serif is metric-compatible with Times New Roman
import subprocess, sys
try:
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-liberation'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except Exception as e:
    print('font install skipped', e)


In [ ]:

# ============ 0. SETUP ============
import os, math, warnings
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings('ignore')
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception:
    pass

PATH = '/content/drive/MyDrive/Obj2_LULC_GWP/'              # same folder as the earlier notebooks
REP  = globals().get('REP', os.path.join(PATH, 'revised_outputs_v2_REPORT'))
OUT  = os.path.join(REP, 'manuscript_figs_tables'); os.makedirs(OUT, exist_ok=True)
PRIMARY = 'anomaly'                                          # within-year standardised features (primary configuration)

hc   = pd.read_csv(os.path.join(REP, 'hindcast_metrics_all.csv'))
bc   = pd.read_csv(os.path.join(REP, 'bootstrap_contrasts.csv'))
fc   = pd.read_csv(os.path.join(REP, 'forecast_change_all.csv'))
print('hindcast rows', len(hc), '| bootstrap contrasts', len(bc), '| forecast rows', len(fc))
print('seeds per mode:', hc[hc.model != 'Persistence'].groupby('mode').seed.nunique().to_dict())

# ---------- style: figures are drawn at their FINAL printed size (6.3 in = 16 cm column width) so that text is 8.5-9.5 pt in Word at 100 % zoom ----------
import matplotlib.font_manager as fm
for _f in fm.findSystemFonts():
    if 'iberation' in _f and 'Serif' in _f: fm.fontManager.addfont(_f)
SERIF = [f for f in ['Times New Roman', 'Liberation Serif', 'Tinos', 'TeX Gyre Termes', 'FreeSerif', 'DejaVu Serif'] if f in {x.name for x in fm.fontManager.ttflist}]
print('figure font:', SERIF[0] if SERIF else 'default')
mpl.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.family': 'serif', 'font.serif': SERIF or ['DejaVu Serif'], 'mathtext.fontset': 'stix',
                     'font.size': 9, 'axes.titlesize': 9.5, 'axes.labelsize': 9, 'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5, 'legend.fontsize': 8.5,
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.25, 'grid.linewidth': 0.5,
                     'axes.linewidth': 0.7, 'xtick.major.width': 0.7, 'ytick.major.width': 0.7, 'lines.linewidth': 1.2, 'legend.frameon': False, 'pdf.fonttype': 42})
W = 6.3          # figure width in inches (= 16 cm, the text width of the manuscript)
COL = {'Persistence': '#6B6B6B', 'CA-ANN-Markov': '#0072B2', 'CNN-Markov': '#D55E00', 'CNN-argmax': '#009E73'}   # Okabe-Ito, colour-blind safe
LAB = {'Persistence': 'Persistence', 'CA_ANN_Markov': 'CA-ANN-Markov', 'CNN_Markov': 'CNN-Markov', 'CNN_argmax': 'CNN-argmax'}
MODE_LAB = {'anomaly': 'Within-year\nstandardised\n(primary)', 'global': 'Global\nmin-max', 'anomaly_noclim': 'No climate\nlayers', 'anomaly_stateonly': 'State + static\nonly'}
MODELS = ['CA_ANN_Markov', 'CNN_Markov', 'CNN_argmax']
CLASSES = ['Water', 'Forest', 'Agriculture', 'Settlement', 'Fallow']
TABLES = {}

def savefig(fig, name):
    fig.savefig(os.path.join(OUT, f'{name}.png'), dpi=300, bbox_inches='tight', pad_inches=0.03)
    fig.savefig(os.path.join(OUT, f'{name}.pdf'), bbox_inches='tight', pad_inches=0.03)                       # vector copy
    fig.savefig(os.path.join(OUT, f'{name}.tiff'), dpi=600, bbox_inches='tight', pad_inches=0.03, pil_kwargs={'compression': 'tiff_lzw'})   # journal-grade copy
    plt.show(); print('saved', name)
TICK = {'CA_ANN_Markov': 'CA-ANN-\nMarkov', 'CNN_Markov': 'CNN-\nMarkov', 'CNN_argmax': 'CNN-\nargmax', 'Persistence': 'Persis-\ntence'}
def pm(x, d=4):                                   # mean +- SD string
    x = np.asarray(x, float); return f'{x.mean():.{d}f} \u00b1 {x.std(ddof=1) if len(x) > 1 else 0:.{d}f}'
def save_table(df, name):
    TABLES[name] = df; df.to_csv(os.path.join(OUT, f'{name}.csv'), index=False); display(df)
def sel(region='test_blocks', mode=PRIMARY, variant='noshield'):
    d = hc[(hc.region == region) & (hc.variant == variant) & (hc['mode'] == mode)]
    return d
def pers(region='test_blocks'): return hc[(hc.model == 'Persistence') & (hc.region == region)].iloc[0]
P = pers()
print(f"Observed change 2013->2023 in test blocks: {P.change_observed*100:.2f}% of {int(P.n):,} px | persistence OA {P.OA*100:.2f}%")


## 1. Table 3 - out-of-time hindcast skill (calibrated on 2003-2013, predict 2023 from 2013), held-out spatial blocks

In [ ]:

def skill_table(region):
    rows = []
    p = pers(region)
    rows.append({'Model': 'Persistence', 'Seeds': '-', 'OA (%)': f'{p.OA*100:.2f}', 'Kappa': f'{p.Kappa:.4f}', 'Macro-F1': f'{p.MacroF1:.4f}', 'FoM': f'{p.FoM:.4f}',
                 'Quantity disagr.': f'{p.QuantityDisagreement:.4f}', 'Allocation disagr.': f'{p.AllocationDisagreement:.4f}',
                 'F1 Water': f'{p.F1_Water:.4f}', 'F1 Fallow': f'{p.F1_Fallow:.4f}', 'Changed px (%)': f'{p.change_predicted*100:.2f}'})
    for m in MODELS:
        d = sel(region)[lambda x: x.model == m]
        if d.empty: continue
        rows.append({'Model': LAB[m], 'Seeds': len(d), 'OA (%)': pm(d.OA*100, 2), 'Kappa': pm(d.Kappa), 'Macro-F1': pm(d.MacroF1), 'FoM': pm(d.FoM),
                     'Quantity disagr.': pm(d.QuantityDisagreement), 'Allocation disagr.': pm(d.AllocationDisagreement),
                     'F1 Water': pm(d.F1_Water), 'F1 Fallow': pm(d.F1_Fallow), 'Changed px (%)': pm(d.change_predicted*100, 2)})
    return pd.DataFrame(rows)
print(f"Observed change in test blocks: {P.change_observed*100:.2f}%")
save_table(skill_table('test_blocks'), 'Table3_hindcast_skill_test_blocks')


## 2. Figure 5 - skill of each model relative to persistence (per-seed points, mean +- SD)

In [ ]:

def fig5():
    fig, ax = plt.subplots(1, 3, figsize=(W, 2.55))
    metrics = [('OA', 'Overall accuracy (%)', 100), ('FoM', 'Figure of merit', 1), ('Kappa', "Cohen's kappa", 1)]
    rng = np.random.RandomState(1)
    for a, (k, ylab, sc) in zip(ax, metrics):
        a.axhline(P[k] * sc, color=COL['Persistence'], ls='--', lw=1.1, label='Persistence')
        for i, m in enumerate(MODELS):
            d = sel()[lambda x: x.model == m][k].values * sc
            if len(d) == 0: continue
            a.scatter(i + rng.uniform(-0.14, 0.14, len(d)), d, s=16, color=COL[LAB[m]], alpha=0.8, edgecolor='white', linewidth=0.3, zorder=3)
            a.errorbar(i, d.mean(), yerr=d.std(ddof=1), color='black', capsize=3, lw=0.9, zorder=4, marker='_', ms=10)
        a.set_xticks(range(len(MODELS))); a.set_xticklabels([TICK[m] for m in MODELS]); a.set_ylabel(ylab); a.set_xlim(-0.5, len(MODELS) - 0.5)
    ax[0].legend(loc='lower left', handlelength=1.6)
    for a, t in zip(ax, ['(a)', '(b)', '(c)']): a.set_title(t, loc='left', fontweight='bold')
    fig.tight_layout(w_pad=0.8); savefig(fig, 'Fig5_hindcast_skill_vs_persistence')
fig5()


## 3. Table 4 and Figure 6 - block-bootstrap contrasts (2000 resamples of 64-px blocks; ensemble-vote maps; all valid blocks)

In [ ]:

PRIM = [('CA_ANN_Markov', 'Persistence'), ('CNN_Markov', 'Persistence'), ('CNN_argmax', 'Persistence'),
        ('CNN_Markov', 'CA_ANN_Markov'), ('CNN_argmax', 'CA_ANN_Markov'), ('CNN_Markov', 'CNN_argmax')]
def get_c(a, b, mode=PRIMARY, var='noshield'):
    A = f'{mode}/{a}/{var}'; B = '-/Persistence/noshield' if b == 'Persistence' else f'{mode}/{b}/{var}'
    r = bc[(bc.A == A) & (bc.B == B)]; return None if r.empty else r.iloc[0]
rows = []
for a, b in PRIM:
    r = get_c(a, b)
    if r is None: continue
    rows.append({'Contrast (A - B)': f'{LAB[a]} - {LAB[b]}', 'dOA (pp)': f'{r.dOA*100:+.2f}', 'OA 95% CI (pp)': f'[{r.OA_lo*100:+.2f}, {r.OA_hi*100:+.2f}]',
                 'dFoM': f'{r.dFoM:+.4f}', 'FoM 95% CI': f'[{r.FoM_lo:+.4f}, {r.FoM_hi:+.4f}]',
                 'OA excl. 0': (r.OA_lo > 0) or (r.OA_hi < 0), 'FoM excl. 0': (r.FoM_lo > 0) or (r.FoM_hi < 0)})
save_table(pd.DataFrame(rows), 'Table4_bootstrap_contrasts_primary')

def forest(ax, items, key, scale, xlabel, title):
    ys = np.arange(len(items))[::-1]
    for y, (lab, r, c) in zip(ys, items):
        lo, hi, m = r[f'{key}_lo'] * scale, r[f'{key}_hi'] * scale, r[f'd{key}'] * scale
        ax.plot([lo, hi], [y, y], color=c, lw=2.0, solid_capstyle='round'); ax.scatter(m, y, color=c, s=20, zorder=3, edgecolor='white', linewidth=0.4)
    ax.axvline(0, color='black', lw=0.8); ax.set_yticks(ys); ax.set_yticklabels([i[0] for i in items]); ax.set_xlabel(xlabel); ax.set_title(title, loc='left', fontweight='bold')
    ax.set_ylim(-0.6, len(items) - 0.4)
def fig6():
    items = []
    for a, b in PRIM:
        r = get_c(a, b)
        if r is not None: items.append((f'{LAB[a]} − {LAB[b]}', r, COL[LAB[a]]))
    fig, ax = plt.subplots(2, 1, figsize=(W, 4.1))
    forest(ax[0], items, 'OA', 100, 'Difference in overall accuracy (percentage points)', '(a)')
    forest(ax[1], items, 'FoM', 1, 'Difference in figure of merit', '(b)')
    fig.tight_layout(h_pad=1.0); savefig(fig, 'Fig6_bootstrap_forest_plot')
fig6()


## 4. Figure 7 - where the error comes from: quantity vs allocation disagreement, and composition of change pixels

In [ ]:

def fig7(region='test_blocks'):
    p = pers(region); order = ['Persistence'] + MODELS
    qd, ad, comp = {}, {}, {}
    qd['Persistence'], ad['Persistence'] = p.QuantityDisagreement, p.AllocationDisagreement
    comp['Persistence'] = np.array([0, p.misses, 0, 0], float)
    for m in MODELS:
        d = sel(region)[lambda x: x.model == m]
        if d.empty: continue
        qd[m], ad[m] = d.QuantityDisagreement.mean(), d.AllocationDisagreement.mean()
        comp[m] = np.array([d.hits.mean(), d.misses.mean(), d.wrong_hits.mean(), d.false_alarms.mean()])
    order = [o for o in order if o in qd]; lab = [TICK[o] for o in order]
    fig, ax = plt.subplots(1, 2, figsize=(W, 3.15))
    x = np.arange(len(order))
    ax[0].bar(x, [qd[o] * 100 for o in order], color='#E69F00', label='Quantity')
    ax[0].bar(x, [ad[o] * 100 for o in order], bottom=[qd[o] * 100 for o in order], color='#56B4E9', label='Allocation')
    for i, o in enumerate(order): ax[0].text(i, (qd[o] + ad[o]) * 100 + 0.12, f'{(qd[o]+ad[o])*100:.2f}', ha='center', fontsize=8.5)
    ax[0].set_xticks(x); ax[0].set_xticklabels(lab); ax[0].set_ylabel('Total disagreement (% of pixels)'); ax[0].legend(loc='upper left', title='Disagreement', title_fontsize=8.5)
    ax[0].set_title('(a)', loc='left', fontweight='bold'); ax[0].set_ylim(0, max((qd[o] + ad[o]) * 100 for o in order) * 1.32)
    cols = ['#009E73', '#CC79A7', '#D55E00', '#F0E442']; names = ['Hits', 'Misses', 'Wrong hits', 'False alarms']
    tot = {o: comp[o].sum() for o in order}; bottom = np.zeros(len(order))
    for j in range(4):
        v = np.array([comp[o][j] / tot[o] * 100 for o in order]); ax[1].bar(x, v, bottom=bottom, color=cols[j], label=names[j]); bottom += v
    ax[1].set_xticks(x); ax[1].set_xticklabels(lab); ax[1].set_ylabel('Share of change-related pixels (%)')
    ax[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2, columnspacing=1.0, handlelength=1.2); ax[1].set_title('(b)', loc='left', fontweight='bold')
    fig.tight_layout(w_pad=1.0); savefig(fig, 'Fig7_error_decomposition')
    tb = pd.DataFrame({'Model': [LAB[o] for o in order], 'Quantity disagr. (%)': [qd[o]*100 for o in order], 'Allocation disagr. (%)': [ad[o]*100 for o in order],
                       'Hits (px)': [comp[o][0] for o in order], 'Misses (px)': [comp[o][1] for o in order], 'Wrong hits (px)': [comp[o][2] for o in order], 'False alarms (px)': [comp[o][3] for o in order]}).round(2)
    save_table(tb, 'Table_S2_change_pixel_composition')
fig7()


## 5. Table 5 and Figure 8 - ablations: feature scaling and covariate sets

In [ ]:

ABL = ['anomaly', 'global', 'anomaly_noclim', 'anomaly_stateonly']
def fig8():
    fig, ax = plt.subplots(2, 1, figsize=(W, 5.3), gridspec_kw={'height_ratios': [1, 1.15]})
    w = 0.30; x = np.arange(len(ABL))
    for j, m in enumerate(['CA_ANN_Markov', 'CNN_Markov']):
        mu, sd = [], []
        for md_ in ABL:
            d = sel(mode=md_)[lambda t: t.model == m].FoM.values; mu.append(d.mean() if len(d) else np.nan); sd.append(d.std(ddof=1) if len(d) > 1 else 0)
        ax[0].bar(x + (j - 0.5) * w * 1.08, mu, w, yerr=sd, capsize=3, color=COL[LAB[m]], label=LAB[m], error_kw={'lw': 0.9})
    ax[0].set_xticks(x); ax[0].set_xticklabels([MODE_LAB[a] for a in ABL]); ax[0].set_ylabel('Figure of merit'); ax[0].legend(loc='upper right'); ax[0].set_title('(a)', loc='left', fontweight='bold')
    SHORT = {'global': 'global scaling', 'anomaly_noclim': 'no climate layers', 'anomaly_stateonly': 'state + static only'}
    items = []
    for m in ['CNN_Markov', 'CA_ANN_Markov']:
        for other in ['global', 'anomaly_noclim', 'anomaly_stateonly']:
            r = bc[(bc.A == f'anomaly/{m}/noshield') & (bc.B == f'{other}/{m}/noshield')]
            if len(r): items.append((f'{LAB[m]}:\n{SHORT[other]}', r.iloc[0], COL[LAB[m]]))
    forest(ax[1], items, 'FoM', 1, 'Difference in figure of merit (primary − ablation)', '(b)')
    fig.tight_layout(h_pad=1.2); savefig(fig, 'Fig8_ablations')
fig8()

rows = []
for m in MODELS:
    for md_ in ABL:
        d = sel(mode=md_)[lambda t: t.model == m]
        if d.empty: continue
        r = bc[(bc.A == f'anomaly/{m}/noshield') & (bc.B == f'{md_}/{m}/noshield')]
        rows.append({'Model': LAB[m], 'Configuration': MODE_LAB[md_].replace('\n', ' '), 'Seeds': len(d), 'OA (%)': pm(d.OA*100, 2), 'FoM': pm(d.FoM),
                     'dOA primary-config (pp) [95% CI]': ('ref.' if md_ == 'anomaly' else f'{r.iloc[0].dOA*100:+.2f} [{r.iloc[0].OA_lo*100:+.2f}, {r.iloc[0].OA_hi*100:+.2f}]') if (md_ == 'anomaly' or len(r)) else 'n/a',
                     'dFoM primary-config [95% CI]': ('ref.' if md_ == 'anomaly' else f'{r.iloc[0].dFoM:+.4f} [{r.iloc[0].FoM_lo:+.4f}, {r.iloc[0].FoM_hi:+.4f}]') if (md_ == 'anomaly' or len(r)) else 'n/a'})
save_table(pd.DataFrame(rows), 'Table5_ablations')


## 6. Table 6 and Figure 9 - does the CNN locate Fallow -> Forest better than chance?
The base rate (share of 2013 Fallow pixels that truly became Forest) is bracketed here from the persistence row alone: persistence precision for Fallow equals the share of 2013 Fallow that stays Fallow, so the true F2F share cannot exceed `1 - precision`; and it cannot be below F2F / (all changed pixels). Section 8 computes it exactly.

In [ ]:

def f2f_bounds(region='test_blocks'):
    p = pers(region)
    prec = p.F1_Fallow * p.R_Fallow / (2 * p.R_Fallow - p.F1_Fallow)          # persistence precision for Fallow = P(stay Fallow | Fallow in 2013)
    upper = 1 - prec                                                           # F2F is a subset of 'leaves Fallow'
    n_fallow13 = None
    # lower bound: Fallow-leavers cannot exceed all changed pixels  ->  N_fallow13 <= misses / (1 - prec)
    nmax = p.misses / (1 - prec); lower = p.F2F_true_px / nmax
    return lower, upper, prec, int(p.F2F_true_px)
lo_b, hi_b, prec_p, f2f_true = f2f_bounds()
print(f'Base-rate bounds for Fallow->Forest in test blocks: {lo_b*100:.1f}% - {hi_b*100:.1f}%  (observed F2F px = {f2f_true})')

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph = k / n; den = 1 + z**2 / n; c = (ph + z**2 / (2 * n)) / den; h = z * math.sqrt(ph * (1 - ph) / n + z**2 / (4 * n**2)) / den; return c - h, c + h
rows = []
for m in ['CA_ANN_Markov', 'CNN_Markov']:
    d = sel()[lambda t: t.model == m]
    pr = (d.F2F_hit_px / d.F2F_pred_px).values; k, n = d.F2F_hit_px.mean(), d.F2F_pred_px.mean(); lo, hi = wilson(k, n)
    rows.append({'Model': LAB[m], 'Seeds': len(d), 'Predicted F2F (px)': pm(d.F2F_pred_px, 0), 'Correct (px)': pm(d.F2F_hit_px, 0), 'Hit precision (%)': pm(pr*100, 1),
                 'Wilson 95% CI on mean counts (%)': f'[{lo*100:.1f}, {hi*100:.1f}]', 'Recall of observed F2F (%)': f'{d.F2F_hit_px.mean()/f2f_true*100:.1f}',
                 'Lift over upper base-rate bound': f'{(pr.mean())/hi_b:.2f}'})
tb6 = pd.DataFrame(rows); tb6['Base-rate range (%)'] = f'{lo_b*100:.1f} - {hi_b*100:.1f}'
save_table(tb6, 'Table6_fallow_to_forest_precision')

def fig9():
    fig, ax = plt.subplots(figsize=(3.5, 3.0)); rng = np.random.RandomState(3)
    ax.axhspan(lo_b * 100, hi_b * 100, color='#BBBBBB', alpha=0.45, label='Possible chance range')
    for i, m in enumerate(['CA_ANN_Markov', 'CNN_Markov']):
        d = sel()[lambda t: t.model == m]; pr = (d.F2F_hit_px / d.F2F_pred_px).values * 100
        ax.scatter(i + rng.uniform(-0.1, 0.1, len(pr)), pr, color=COL[LAB[m]], s=20, edgecolor='white', linewidth=0.3, zorder=3)
        k, n = d.F2F_hit_px.mean(), d.F2F_pred_px.mean(); lo, hi = wilson(k, n)
        ax.errorbar(i + 0.24, pr.mean(), yerr=[[pr.mean() - lo * 100], [hi * 100 - pr.mean()]], color='black', capsize=3, marker='o', ms=3.5, lw=0.9)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['CA-ANN-\nMarkov', 'CNN-\nMarkov']); ax.set_xlim(-0.5, 1.7); ax.set_ylabel('Hit precision, Fallow → Forest (%)')
    ax.legend(loc='upper left'); ax.set_ylim(0, 65)
    fig.tight_layout(); savefig(fig, 'Fig9_fallow_to_forest_precision')
fig9()


## 7. Table 7 and Figure 11 - 2033 scenario projections and what drives them

In [ ]:

fcp = fc[(fc['mode'] == PRIMARY) & (fc.variant == 'noshield')].copy()
dcols = ['d_Water', 'd_Forest', 'd_Agriculture', 'd_Settlement', 'd_Fallow']
SC = ['T1', 'Mean', 'T2']
rows = []
for m in ['CA_ANN_Markov', 'CNN_Markov']:
    for s in SC:
        d = fcp[(fcp.model == m) & (fcp.scenario == s)]
        if d.empty: continue
        rows.append({'Model': LAB[m], 'Markov rates': {'T1': '2003-2013', 'Mean': 'mean of both', 'T2': '2013-2023'}[s], 'Seeds': len(d),
                     **{c.replace('d_', 'd'): pm(d[c], 2) for c in dcols}, 'Pixels changed (%)': pm(d.pct_changed, 2), 'New steep-slope water (px)': pm(d.new_water_steep_px, 1)})
save_table(pd.DataFrame(rows), 'Table7_forecast_2033_scenarios_km2')

# variance partition: share of between-seed + between-scenario variance in dForest that is due to the scenario (eta-squared)
eta = []
for m in ['CA_ANN_Markov', 'CNN_Markov']:
    for c in dcols:
        d = fcp[fcp.model == m]; g = [d[d.scenario == s][c].values for s in SC if len(d[d.scenario == s])]
        allv = np.concatenate(g); ssb = sum(len(x) * (x.mean() - allv.mean())**2 for x in g); sst = ((allv - allv.mean())**2).sum()
        F, pval = stats.f_oneway(*g)
        eta.append({'Model': LAB[m], 'Class': c.replace('d_', ''), 'eta2 (scenario)': round(ssb / sst, 4), 'F': round(F, 1), 'p': f'{pval:.1e}',
                    'Scenario range (km2)': round(max(x.mean() for x in g) - min(x.mean() for x in g), 2), 'Mean seed SD (km2)': round(np.mean([x.std(ddof=1) for x in g]), 2)})
save_table(pd.DataFrame(eta), 'Table_S3_scenario_vs_seed_variance')

def fig11():
    fig = plt.figure(figsize=(W, 5.6)); gs = fig.add_gridspec(2, 2, height_ratios=[1, 0.9])
    axA, axB, axC = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, :])
    cls = ['Water', 'Forest', 'Agri.', 'Settl.', 'Fallow']
    for a, m, t in [(axA, 'CNN_Markov', '(a) CNN-Markov'), (axB, 'CA_ANN_Markov', '(b) CA-ANN-Markov')]:
        w = 0.26
        for j, (s_, c) in enumerate(zip(SC, ['#56B4E9', '#E69F00', '#D55E00'])):
            d = fcp[(fcp.model == m) & (fcp.scenario == s_)]
            if d.empty: continue
            a.bar(np.arange(5) + (j - 1) * w, [d[k].mean() for k in dcols], w, yerr=[d[k].std(ddof=1) for k in dcols], capsize=1.5, color=c, label={'T1': 'T1 rates (2003–13)', 'Mean': 'Mean rates', 'T2': 'T2 rates (2013–23)'}[s_], error_kw={'lw': 0.7})
        a.axhline(0, color='black', lw=0.7); a.set_xticks(range(5)); a.set_xticklabels(cls); a.set_title(t, loc='left', fontweight='bold'); a.set_ylim(-48, 48)
    axA.set_ylabel('Change 2023→2033 (km²)'); axB.set_yticklabels([])
    axA.legend(loc='upper right', fontsize=8, handlelength=1.2)
    w = 0.38
    for j, (md_, c, l) in enumerate([('anomaly', '#009E73', 'Within-year standardised'), ('global', '#CC79A7', 'Global min-max')]):
        d = fc[(fc['mode'] == md_) & (fc.model == 'CNN_argmax') & (fc.variant == 'noshield')]
        if d.empty: continue
        axC.bar(np.arange(5) + (j - 0.5) * w, [d[k].mean() for k in dcols], w, yerr=[d[k].std(ddof=1) for k in dcols], capsize=2, color=c, label=l, error_kw={'lw': 0.8})
    axC.axhline(0, color='black', lw=0.7); axC.set_xticks(range(5)); axC.set_xticklabels(['Water', 'Forest', 'Agriculture', 'Settlement', 'Fallow']); axC.set_ylabel('Change 2023→2033 (km²)')
    axC.legend(loc='lower left'); axC.set_title('(c) CNN-argmax: effect of year-label confounding on the forecast', loc='left', fontweight='bold')
    fig.tight_layout(h_pad=1.0, w_pad=0.8); savefig(fig, 'Fig11_forecast_scenarios')
fig11()


## 8. Supplementary Table S1 (all valid blocks) and Excel workbook

In [ ]:

save_table(skill_table('all_valid'), 'Table_S1_hindcast_skill_all_valid_blocks')

# Table S4 - the 'historical stability shield' (restore the 2013 class wherever the class did not change 2003->2013) is persistence on those pixels
rows = []
for m in MODELS:
    for var in ['noshield', 'shield']:
        d = sel(variant=var)[lambda t: t.model == m]
        if d.empty: continue
        rows.append({'Model': LAB[m], 'Variant': 'with shield' if var == 'shield' else 'without shield', 'Seeds': len(d), 'OA (%)': pm(d.OA*100, 2), 'FoM': pm(d.FoM), 'Changed px (%)': pm(d.change_predicted*100, 2)})
rows.append({'Model': 'Persistence', 'Variant': '-', 'Seeds': '-', 'OA (%)': f'{P.OA*100:.2f}', 'FoM': f'{P.FoM:.4f}', 'Changed px (%)': '0.00'})
save_table(pd.DataFrame(rows), 'Table_S4_stability_shield_variant')
try:
    with pd.ExcelWriter(os.path.join(OUT, 'all_tables.xlsx')) as xw:
        for k, v in TABLES.items(): v.to_excel(xw, sheet_name=k[:31], index=False)
    print('Excel workbook written')
except Exception as e:
    print('Excel export skipped:', e)


## 9. OPTIONAL - exact Fallow -> Forest base rate and block-bootstrap interval (needs the rasters; CPU, a few minutes)
Replaces the bracketed base-rate range in Table 6 / Fig. 9 by the exact value and gives block-bootstrap CIs for hit precision, which respects spatial autocorrelation (the Wilson interval above does not).
Set `RUN_RASTER_CHECK = True` to run.

In [ ]:

RUN_RASTER_CHECK = False
SEEDS_USED = [0, 1, 2, 3, 4]

def f2f_block_boot(prev, true, preds, block_id, mask, fallow=4, forest=1, n=2000, seed=0):
    """prev/true: 2013/2023 index maps; preds: dict name -> index map. Returns exact base rate + block-bootstrap CIs of hit precision and of the lift over base rate."""
    ids = block_id[mask]; ub, inv = np.unique(ids, return_inverse=True); NB = len(ub)
    q, t = prev[mask], true[mask]
    isF = q == fallow
    cnt = lambda v: np.bincount(inv, weights=v.astype(float), minlength=NB)
    n_f, n_f2f = cnt(isF), cnt(isF & (t == forest)); out = {'base': (n_f2f.sum() / n_f.sum())}; B = {}
    for k, p in preds.items():
        pm_ = p[mask]; B[k] = (cnt(isF & (pm_ == forest)), cnt(isF & (pm_ == forest) & (t == forest)))
    rng = np.random.RandomState(seed); res = {k: [] for k in preds}; base_s = []
    for _ in range(n):
        s = rng.randint(0, NB, NB); b = n_f2f[s].sum() / max(1, n_f[s].sum()); base_s.append(b)
        for k in preds: res[k].append(B[k][1][s].sum() / max(1, B[k][0][s].sum()))
    for k in preds:
        r = np.array(res[k]); out[k] = dict(precision=B[k][1].sum() / max(1, B[k][0].sum()), lo=np.percentile(r, 2.5), hi=np.percentile(r, 97.5),
                                            lift=B[k][1].sum() / max(1, B[k][0].sum()) / out['base'], lift_lo=np.percentile(r / np.array(base_s), 2.5), lift_hi=np.percentile(r / np.array(base_s), 97.5),
                                            n_pred=int(B[k][0].sum()), n_hit=int(B[k][1].sum()))
    out['base_lo'], out['base_hi'] = np.percentile(base_s, [2.5, 97.5]); return out

if RUN_RASTER_CHECK:
    import rasterio
    CV = [1, 2, 3, 4, 5]
    def rd(p):
        with rasterio.open(p) as s: a = s.read(1)
        o = np.full(a.shape, -1, np.int8)
        for k, v in enumerate(CV): o[a == v] = k
        return o
    L = {y: rd(os.path.join(PATH, f'LULC_{y}.tif')) for y in ['2003', '2013', '2023']}
    VALID = np.all([L[y] >= 0 for y in L], axis=0); H, W = VALID.shape; BLOCK = 64
    nbr, nbc = math.ceil(H / BLOCK), math.ceil(W / BLOCK)
    has = np.array([[VALID[i*BLOCK:(i+1)*BLOCK, j*BLOCK:(j+1)*BLOCK].any() for j in range(nbc)] for i in range(nbr)])
    ids = np.argwhere(has); rg = np.random.RandomState(42); rg.shuffle(ids)
    n_te, n_va = int(round(0.15 * len(ids))), int(round(0.15 * len(ids))); lab = np.full((nbr, nbc), -1, np.int8)
    for n_, (i, j) in enumerate(ids): lab[i, j] = 2 if n_ < n_te else (1 if n_ < n_te + n_va else 0)
    SPLIT = np.where(VALID, np.repeat(np.repeat(lab, BLOCK, 0), BLOCK, 1)[:H, :W], -1)
    BLOCK_ID = np.repeat(np.repeat(np.arange(nbr*nbc).reshape(nbr, nbc), BLOCK, 0), BLOCK, 1)[:H, :W]
    def vote(maps):
        st = np.stack(maps); return np.where(VALID, np.stack([(st == k).sum(0) for k in range(5)]).argmax(0), -1).astype(np.int8)
    preds = {}
    for tag in ['CA_ANN_Markov', 'CNN_Markov']:
        fol = os.path.join(PATH, 'revised_outputs_v2_anomaly')
        maps = [rd(os.path.join(fol, f'{tag}_hind2023_seed{s}.tif')) for s in SEEDS_USED if os.path.exists(os.path.join(fol, f'{tag}_hind2023_seed{s}.tif'))]
        if maps: preds[LAB[tag]] = vote(maps)
    rows = []
    for reg, mask in [('test_blocks', SPLIT == 2), ('all_valid', VALID)]:
        r = f2f_block_boot(L['2013'], L['2023'], preds, BLOCK_ID, mask)
        for k in preds:
            rows.append({'Region': reg, 'Model (ensemble vote)': k, 'Exact base rate (%)': f"{r['base']*100:.1f} [{r['base_lo']*100:.1f}, {r['base_hi']*100:.1f}]", 'Predicted F2F (px)': r[k]['n_pred'], 'Correct (px)': r[k]['n_hit'],
                         'Hit precision (%)': f"{r[k]['precision']*100:.1f} [{r[k]['lo']*100:.1f}, {r[k]['hi']*100:.1f}]", 'Lift over base rate': f"{r[k]['lift']:.2f} [{r[k]['lift_lo']:.2f}, {r[k]['lift_hi']:.2f}]"})
    save_table(pd.DataFrame(rows), 'Table6b_fallow_to_forest_exact_base_rate_block_bootstrap')
else:
    print('Skipped (set RUN_RASTER_CHECK = True to run).')
